In [1]:
import pandas as pd
import sqlite3

from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent

RAW_DATA_PATH = (
    BASE_DIR
    / "data"
    / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

PROCESSED_DIR = (
    BASE_DIR
    / "data"
    / "processed"
)

DB_PATH = (
    PROCESSED_DIR
    / "customer_intelligence.db"
)

print("Raw dataset:", RAW_DATA_PATH)
print("Database:", DB_PATH)

Raw dataset: /home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
Database: /home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /data/processed/customer_intelligence.db


In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (7043, 21)


In [4]:
df["TotalCharges"].dtype

<StringDtype(storage='python', na_value=nan)>

In [5]:
total_charges_numeric = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print(
    "Values converted to NaN:",
    total_charges_numeric.isna().sum()
)

Values converted to NaN: 11


In [6]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print(df["TotalCharges"].dtype)

float64


In [8]:
# Create the SQLite database

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

connection = sqlite3.connect(DB_PATH)

print("SQLite database connected successfully")

SQLite database connected successfully


In [9]:
# Load customer data into SQLite    

df.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

print("Customer data loaded into SQLite")

Customer data loaded into SQLite


In [10]:
# Verify the table

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    connection
)

tables

,name
0,customers


In [11]:
# View the first records using SQL

query = """
SELECT *
FROM customers
LIMIT 5;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
# Total customers

query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,total_customers
0,7043


In [13]:
# Total churned customers

query = """
SELECT COUNT(*) AS churned_customers
FROM customers
WHERE Churn = 'Yes';
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,churned_customers
0,1869


In [14]:
# Total non-churned customers

query = """
SELECT COUNT(*) AS retained_customers
FROM customers
WHERE Churn = 'No';
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,retained_customers
0,5174


In [15]:
# Overall churn rate

query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate
FROM customers;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,total_customers,churned_customers,churn_rate
0,7043,1869,26.54


In [16]:
# Churn by contract type

query = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate
    
FROM customers

GROUP BY Contract

ORDER BY churn_rate DESC;
"""

contract_analysis = pd.read_sql_query(
    query,
    connection
)

contract_analysis

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [17]:
# Churn by payment method

query = """
SELECT
    PaymentMethod,
    COUNT(*) AS total_customers,
    
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate
    
FROM customers

GROUP BY PaymentMethod

ORDER BY churn_rate DESC;
"""

payment_analysis = pd.read_sql_query(
    query,
    connection
)

payment_analysis

,PaymentMethod,total_customers,churned_customers,churn_rate
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


In [18]:
# Churn by internet service

query = """
SELECT
    InternetService,
    COUNT(*) AS total_customers,
    
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate
    
FROM customers

GROUP BY InternetService

ORDER BY churn_rate DESC;
"""

internet_analysis = pd.read_sql_query(
    query,
    connection
)

internet_analysis

,InternetService,total_customers,churned_customers,churn_rate
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [19]:
# Average monthly charges

query = """
SELECT
    ROUND(AVG(MonthlyCharges), 2) AS average_monthly_charges
FROM customers;
"""

pd.read_sql_query(
    query,
    connection
)

,average_monthly_charges
0,64.76


In [20]:
# Average total charges

query = """
SELECT
    ROUND(AVG(TotalCharges), 2) AS average_total_charges
FROM customers;
"""

pd.read_sql_query(
    query,
    connection
)

,average_total_charges
0,2283.3


In [21]:
# Monthly charges by churn

query = """
SELECT
    Churn,
    COUNT(*) AS customers,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges
FROM customers
GROUP BY Churn;
"""

monthly_charges_churn = pd.read_sql_query(
    query,
    connection
)

monthly_charges_churn

,Churn,customers,avg_monthly_charges
0,No,5174,61.27
1,Yes,1869,74.44


In [22]:
# Total charges by churn

query = """
SELECT
    Churn,
    COUNT(*) AS customers,
    ROUND(AVG(TotalCharges), 2) AS avg_total_charges
FROM customers
GROUP BY Churn;
"""

pd.read_sql_query(
    query,
    connection
)

,Churn,customers,avg_total_charges
0,No,5174,2555.34
1,Yes,1869,1531.80


In [23]:
# High-value customers

high_value_threshold = df["MonthlyCharges"].quantile(0.75)

print(
    "High-value threshold:",
    round(high_value_threshold, 2)
)

High-value threshold: 89.85


In [24]:
query = f"""
SELECT
    customerID,
    Contract,
    tenure,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM customers
WHERE MonthlyCharges >= {high_value_threshold}
ORDER BY MonthlyCharges DESC;
"""

high_value_customers = pd.read_sql_query(
    query,
    connection
)

high_value_customers.head(10)

,customerID,Contract,tenure,MonthlyCharges,TotalCharges,Churn
0,7569-NMZYQ,Two year,72,118.75,8672.45,No
1,8984-HPEMB,Two year,71,118.65,8477.60,No
2,5989-AXPUC,Two year,68,118.60,7990.05,No
3,5734-EJKXG,One year,61,118.60,7365.70,No
4,8199-ZLLSA,One year,67,118.35,7804.15,Yes
5,9924-JPRMC,Two year,72,118.20,8547.15,No
6,2889-FPWRM,One year,72,117.80,8684.80,Yes
7,3810-DVDQQ,Two year,72,117.60,8308.90,No
8,9739-JLPQJ,Two year,72,117.50,8670.10,No
9,2302-ANTDP,Month-to-month,48,117.45,5438.90,Yes


In [ ]:
# High-value customer churn

query = f"""
SELECT
    COUNT(*) AS high_value_customers,
    
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_high_value_customers,
    
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS high_value_churn_rate

FROM customers

WHERE MonthlyCharges >= {high_value_threshold};
"""

pd.read_sql_query(
    query,
    connection
)

,high_value_customers,churned_high_value_customers,high_value_churn_rate
0,1771,580,32.75


In [ ]:
# Long-tenure customer churn    

long_tenure_threshold = df["tenure"].quantile(0.75)

print(
    "Long-tenure threshold:",
    round(long_tenure_threshold, 2),
    "months"
)

Long-tenure threshold: 55.0 months


In [27]:
query = f"""
SELECT
    customerID,
    tenure,
    Contract,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM customers
WHERE tenure >= {long_tenure_threshold}
ORDER BY tenure DESC;
"""

long_tenure_customers = pd.read_sql_query(
    query,
    connection
)

long_tenure_customers.head(10)

,customerID,tenure,Contract,MonthlyCharges,TotalCharges,Churn
0,5248-YGIJN,72,Two year,90.25,6369.45,No
1,6234-RAAPL,72,Two year,99.90,7251.70,No
2,5954-BDFSG,72,Two year,107.50,7853.70,No
3,0526-SXDJP,72,Two year,42.10,2962.00,No
4,9848-JQJTX,72,Two year,100.90,7459.05,No
5,6728-DKUCO,72,One year,104.15,7303.05,No
6,2848-YXSMW,72,Two year,19.40,1363.25,No
7,6734-PSBAW,72,Two year,23.55,1723.95,No
8,3146-MSEGF,72,Two year,88.05,6425.65,No
9,5997-OPVFA,72,Two year,89.05,6254.45,No


In [28]:
# Long-tenure customer churn    

query = f"""
SELECT
    COUNT(*) AS long_tenure_customers,
    
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_long_tenure_customers,
    
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate

FROM customers

WHERE tenure >= {long_tenure_threshold};
"""

pd.read_sql_query(
    query,
    connection
)

,long_tenure_customers,churned_long_tenure_customers,churn_rate
0,1819,145,7.97


In [29]:
# High-risk customer groups

query = """
SELECT
    Contract,
    InternetService,
    COUNT(*) AS total_customers,
    
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate

FROM customers

GROUP BY
    Contract,
    InternetService

HAVING COUNT(*) >= 20

ORDER BY churn_rate DESC;
"""

risk_groups = pd.read_sql_query(
    query,
    connection
)

risk_groups

,Contract,InternetService,total_customers,churned_customers,churn_rate
0,Month-to-month,Fiber optic,2128,1162,54.61
1,Month-to-month,DSL,1223,394,32.22
2,One year,Fiber optic,539,104,19.29
3,Month-to-month,No,524,99,18.89
4,One year,DSL,570,53,9.30
5,Two year,Fiber optic,429,31,7.23
6,One year,No,364,9,2.47
7,Two year,DSL,628,12,1.91
8,Two year,No,638,5,0.78


In [30]:
# Customer service segments

query = """
SELECT
    InternetService,
    PhoneService,
    COUNT(*) AS customer_count
FROM customers

GROUP BY
    InternetService,
    PhoneService

ORDER BY customer_count DESC;
"""

service_segments = pd.read_sql_query(
    query,
    connection
)

service_segments

,InternetService,PhoneService,customer_count
0,Fiber optic,Yes,3096
1,DSL,Yes,1739
2,No,Yes,1526
3,DSL,No,682


In [31]:
# Revenue-related analysis

query = """
SELECT
    ROUND(SUM(MonthlyCharges), 2) AS total_monthly_revenue,
    ROUND(AVG(MonthlyCharges), 2) AS average_monthly_charge
FROM customers;
"""

revenue_analysis = pd.read_sql_query(
    query,
    connection
)

revenue_analysis

,total_monthly_revenue,average_monthly_charge
0,456116.6,64.76


In [32]:
# Revenue by contract

query = """
SELECT
    Contract,
    COUNT(*) AS customers,
    ROUND(SUM(MonthlyCharges), 2) AS monthly_revenue,
    ROUND(AVG(MonthlyCharges), 2) AS average_monthly_charge
FROM customers

GROUP BY Contract

ORDER BY monthly_revenue DESC;
"""

revenue_by_contract = pd.read_sql_query(
    query,
    connection
)

revenue_by_contract

,Contract,customers,monthly_revenue,average_monthly_charge
0,Month-to-month,3875,257294.15,66.40
1,Two year,1695,103005.85,60.77
2,One year,1473,95816.60,65.05


In [33]:
# Churned revenue exposure

query = """
SELECT
    ROUND(
        SUM(
            CASE
                WHEN Churn = 'Yes'
                THEN MonthlyCharges
                ELSE 0
            END
        ),
        2
    ) AS churned_monthly_revenue
FROM customers;
"""

pd.read_sql_query(
    query,
    connection
)

,churned_monthly_revenue
0,139130.85


In [34]:
# Customer lookup query

customer_id = df["customerID"].iloc[0]

query = """
SELECT *
FROM customers
WHERE customerID = ?;
"""

customer = pd.read_sql_query(
    query,
    connection,
    params=(customer_id,)
)

customer

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No


In [ ]:
# Create a reusable SQL analytics function